In [1]:
# 03_classification_power.ipynb
# (A) Classification: can the convergence index identify oversight failure better
# than Stage-1 dispersion alone (ROC / AUC)? (B) Level-2 power: how many boards are
# needed to detect the board-level determinant (escalation authority) of the
# convergence index? Analysis uses boards with >=3 respondents. Figures greyscale;
# no captions embedded.
import numpy as np, pandas as pd
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
from sklearn.metrics import roc_curve, roc_auc_score
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "font.size": 11,
                     "axes.edgecolor": "black", "axes.linewidth": 0.8})
DATA = Path("..")/"data"; FIG = Path("..")/"results"/"figures"; TAB = Path("..")/"results"/"tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
def save(fig, name):
    fig.savefig(FIG/f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG/f"{name}.pdf", bbox_inches="tight"); plt.close(fig)

b = pd.read_csv(DATA/"board_analysis.csv")
bb = b[b.n_resp >= 3].copy()
y = bb.gt_failure.values

# (A) ROC for three classifiers of oversight failure
Xc = sm.add_constant(bb[["disp1", "conv_disp"]])
comb = sm.Logit(y, Xc).fit(disp=0)
scores = {"Stage-1 dispersion": bb.disp1.values,
          "Stage-2 residual (low convergence)": -bb.conv_disp.values,
          "Dispersion + convergence": comb.predict(Xc).values}
styles = ["-", "--", ":"]; aucs = {}
fig, ax = plt.subplots(figsize=(5.0, 4.6))
for (name, sc), ls in zip(scores.items(), styles):
    fpr, tpr, _ = roc_curve(y, sc); a = roc_auc_score(y, sc); aucs[name] = a
    ax.plot(fpr, tpr, color="black", ls=ls, lw=1.6, label=f"{name} (AUC={a:.2f})")
ax.plot([0, 1], [0, 1], color="#999999", lw=0.8)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.legend(frameon=True, edgecolor="black", loc="lower right")
save(fig, "fig04_roc")
pd.DataFrame({"classifier": list(aucs), "AUC": [round(v, 3) for v in aucs.values()]}
             ).to_csv(TAB/"table04_auc.csv", index=False)

# (B) Level-2 power curve: detect escalation effect on convergence across board counts
rng = np.random.default_rng(7)
Ns = [20, 40, 60, 80, 100, 120, 140]; reps = 400; rows = []
for n in Ns:
    hits = 0
    for _ in range(reps):
        samp = bb.sample(n=min(n, len(bb)), replace=True, random_state=int(rng.integers(1_000_000_000)))
        try:
            m = smf.ols("conv_disp ~ escalation + independence + diversity", samp).fit()
            hits += int(m.pvalues.get("escalation", 1.0) < 0.05)
        except Exception:
            pass
    rows.append(dict(n_boards=n, power=hits/reps))
pw = pd.DataFrame(rows); pw.to_csv(TAB/"table05_power.csv", index=False)
fig, ax = plt.subplots(figsize=(5.2, 3.8))
ax.plot(pw.n_boards, pw.power, marker="o", color="#222222", lw=1.4)
ax.axhline(0.8, color="#777777", ls="--", lw=1.0)
ax.set_xlabel("Number of boards (Level-2 units)")
ax.set_ylabel("Power to detect escalation effect"); ax.set_ylim(0, 1.02)
save(fig, "fig05_level2_power")

print("AUCs:", {k: round(v, 3) for k, v in aucs.items()})
print(pw.to_string(index=False))
print("figures: fig04, fig05  | tables: table04, table05")


AUCs: {'Stage-1 dispersion': 0.738, 'Stage-2 residual (low convergence)': 0.695, 'Dispersion + convergence': 0.829}
 n_boards  power
       20 0.6975
       40 0.9675
       60 0.9925
       80 1.0000
      100 1.0000
      120 1.0000
      140 1.0000
figures: fig04, fig05  | tables: table04, table05
